In [1]:
from matplotlib import pyplot as plt
import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
from statistics import mean
import seaborn as sns
import os

import statsmodels.formula.api as sm
from sklearn import linear_model
from sklearn import tree, svm, metrics
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.preprocessing import scale
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

In [2]:
#function to gather the data
def driver_per_race(session):
    total_table = pd.DataFrame(columns = ["Driver", "Team", "Result", "Start Pos", "Finish Pos", "Max Speed", "Avg Speed","DRS active", "Total Laps",
                                          "Soft Percent", "Medium Percent", "Hard Percent", "Intermediate Percent", "Wet Percent", 'total_pit_time'])
    #for time delta
    time_count = pd.Timedelta(0).total_seconds()
    #iterates through each driver for that race
    for driver in session.drivers:
        #breaks for when race is missing/not happened yet
        try:
            code = session.get_driver(driver).Abbreviation
            team = session.get_driver(driver).TeamName
            tele = session.laps.pick_driver(driver).get_telemetry()
            tele = tele[tele['Status'].str.contains("OnTrack")]

            compounds = {'SOFT': 0, 'MEDIUM': 0, 'HARD': 0, 'INTERMEDIATE': 0, 'WET': 0}
            laps = len(session.laps.pick_driver(driver))

            time_count = time_count + session.get_driver(driver)['Time'].total_seconds()
            
            sp_max = tele["Speed"].max()
            sp_avg = tele["Speed"].mean()
            drs_prop = len(tele[tele["DRS"] > 8]) / len(tele["DRS"]) * 100
            start = session.get_driver(driver)['GridPosition']
            end = session.get_driver(driver)['Position']
            storage = [code, team, time_count, start, end, sp_max, sp_avg,drs_prop, laps]
            
            #tire data
            for tire in session.laps.pick_driver(driver)["Compound"]:
                compounds[tire] += 1

            storage = storage + [compounds['SOFT']/laps, compounds['MEDIUM']/laps,
                                 compounds['HARD']/laps, compounds['INTERMEDIATE']/laps, compounds['WET']/laps]
            #pit times
            pit_in = session.laps.pick_driver(driver)['PitInTime'].reset_index(drop=True)
            pit_out = session.laps.pick_driver(driver)['PitOutTime'].reset_index(drop=True)

            temp_array = []
            for number in range(len(pit_in)-1):
                pit_time = pit_out[number + 1] - pit_in[number]
                temp_array.append(pit_time)

            filtered_array = [x for x in temp_array if x is not pd.NaT]

            total_pit_time = sum(filtered_array, pd.Timedelta(0)).total_seconds()
            storage = storage + [total_pit_time]
            total_table = pd.concat([total_table, pd.DataFrame([storage], columns = total_table.columns)], ignore_index = True)
        except Exception as e:
            pass

    max_stints = session.laps.groupby('Driver').agg(Max_Stint=('Stint', 'max')).reset_index()
    total_table = pd.merge(total_table, max_stints, on='Driver')
    return(total_table)

In [4]:
#function to create dicitonary of races for each championship year
def year_data(year):
    rounds = {}
    try:
        for race in range(max(fastf1.get_event_schedule(year).RoundNumber)):
            session = fastf1.get_session(year, race + 1, 'R')
            session.load()
            event_data = driver_per_race(session)
            rounds[str(session.event.EventName)] = event_data
    except Exception as e:
        return(rounds)
    
    return(rounds)

In [ ]:
drivers2022 = year_data(2022)
drivers2023 = year_data(2023)
drivers2024 = year_data(2024)

In [ ]:
#makes csvs for each race and a combined data
directory_path = 'C:/Users/metal/OneDrive/Documents/UU files/data sci/f1 project data/'

#for combined data
driver_data_list = []

year_list = [2022, 2023, 2024]
for i in year_list:
    d_list = f"drivers{i}"
    for item in globals()[d_list]:
        ref = globals()[d_list][item]
        driver_data_list.append(ref)
        csv_name = f"{item} {i}.csv"
        csv_path = os.path.join(directory_path, csv_name)
        ref.to_csv(csv_path, index=False)

master_driver = pd.concat(driver_data_list, ignore_index = True)
csv_name = "total drivers data.csv"
csv_path = os.path.join(directory_path, csv_name)
master_driver.to_csv(csv_path, index=False)

In [149]:
#for easy retrieval of data and normalization
folder_path = 'C:/Users/metal/OneDrive/Documents/UU files/data sci/f1 project data/'

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
csv_list = []

for csv_file in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['Result'])
    df['Max Speed'] = (df['Max Speed'] - df['Max Speed'].median()) / max(abs((df['Max Speed'] - df['Max Speed'].median())))
    df['Avg Speed'] = (df['Avg Speed'] - df['Avg Speed'].median()) / max(abs((df['Avg Speed'] - df['Avg Speed'].median())))
    df['total_pit_time'] = (df['total_pit_time'] - df['total_pit_time'].median()) / max(abs((df['total_pit_time'] - df['total_pit_time'].median())))
    csv_list.append(df)
    
    master_driver = pd.concat(csv_list)

In [ ]:
#Read in British Grand Prix csv using pandas
BritGP2023 = pd.read_csv("f1 data raw/British Grand Prix 2023.csv")

#Creating Correlation Matrix
corr_matrix = BritGP2023.corr(numeric_only=True)

plt.figure(figsize=(10, 8))

heatmap = sns.heatmap(corr_matrix, cmap='inferno', annot=True, vmin=-1, vmax=1)

# Create colorbar directly from the heatmap object
colorbar = heatmap.collections[0].colorbar
colorbar.set_label('Correlation')

plt.xticks(range(len(corr_matrix)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix)), corr_matrix.columns)

plt.title('Correlation Matrix for British Grand Prix 2023')
plt.show()

In [ ]:
#Loop to fix variable names so they pull into the ols function properly
for column in BritGP2023.columns:
    if ' ' in column:
        new_column_name = column.replace(' ', '_')
        BritGP2023.rename(columns={column: new_column_name}, inplace=True)

#Creating and testing a linear regression model based off the variables with the strongest variables to predict Result time.
Result_Time_ols = sm.ols(formula="Result ~ Start_Pos + Avg_Speed", data=BritGP2023).fit()
Result_Time_ols.summary()

In [ ]:
#Loop to fix variable names so they pull into the ols function properly
for column in BritGP2023.columns:
    if ' ' in column:
        new_column_name = column.replace(' ', '_')
        BritGP2023.rename(columns={column: new_column_name}, inplace=True)

#Creating and testing a linear regression model based for Result ~ Avgerage Speed
Result_Time_ols = sm.ols(formula="Result ~ Avg_Speed", data=BritGP2023).fit()
Result_Time_ols.summary()

In [ ]:
pd.plotting.scatter_matrix(BritGP2023, figsize=(10, 10), diagonal='hist');

pd.plotting.scatter_matrix(total_races, figsize=(10, 10), diagonal='hist');

In [ ]:
#Creating a data frame including all races where the Max Speed and Average Speed are normalized
folder_path = "f1 data raw"

# List all files in the folder and filter for CSV files
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
csv_list = []

# Iterate through each CSV file
for csv_file in csv_files:
    # Create the full file path
    file_path = os.path.join(folder_path, csv_file)
    
    # Read the CSV file using pandas
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['Result'])
    df['Max Speed'] = (df['Max Speed'] - df['Max Speed'].median()) / max(abs((df['Max Speed'] - df['Max Speed'].median())))
    df['Avg Speed'] = (df['Avg Speed'] - df['Avg Speed'].median()) / max(abs((df['Avg Speed'] - df['Avg Speed'].median())))
    df['total_pit_time'] = df['total_pit_time'] / df['total_pit_time'].median()
    csv_list.append(df)
    
    # Process the DataFrame as needed
    master_driver = pd.concat(csv_list)
master_driver

In [ ]:
#Creating a histogram for Average Speed before normalizing the variable.
total_races['Avg Speed'].hist()

# Add labels and title
plt.xlabel('Avg Speed')
plt.ylabel('Frequency')
plt.title('Histogram of Avg Speed Before Normalization')

# Show the plot
plt.show()


#Creating a histogram for Average Speed after normalizing the variable.
master_driver['Avg Speed'].hist()

# Add labels and title
plt.xlabel('Avg Speed')
plt.ylabel('Frequency')
plt.title('Histogram of Avg Speed After Normalization')

# Show the plot
plt.show()

In [ ]:
#Creating Correlation Matrix
corr_matrix = master_driver.corr(numeric_only=True)

plt.figure(figsize=(10, 8))

heatmap = sns.heatmap(corr_matrix, cmap='inferno', annot=True, vmin=-1, vmax=1)

# Create colorbar directly from the heatmap object
colorbar = heatmap.collections[0].colorbar
colorbar.set_label('Correlation')

plt.xticks(range(len(corr_matrix)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix)), corr_matrix.columns)

plt.title('Correlation Matrix for All Races')
plt.show()

In [ ]:
#Regression model for Result Time ~ Starting Position + Average Speed

#Loop to fix variable names so they pull into the ols function properly
for column in master_driver.columns:
    if ' ' in column:
        new_column_name = column.replace(' ', '_')
        master_driver.rename(columns={column: new_column_name}, inplace=True)

#Creating and testing a linear regression model based off the variables with the strongest variables to predict Result time.
Result_Time_ols = sm.ols(formula="Result ~ Start_Pos + Avg_Speed", data=master_driver).fit()
Result_Time_ols.summary()

In [ ]:
#Regression model for Finishing Position ~ Starting Position + Average Speed

#Loop to fix variable names so they pull into the ols function properly
for column in master_driver.columns:
    if ' ' in column:
        new_column_name = column.replace(' ', '_')
        master_driver.rename(columns={column: new_column_name}, inplace=True)

#Creating and testing a linear regression model based off the variables with the strongest variables to predict Result time.
Result_Time_ols = sm.ols(formula="Finish_Pos ~ Start_Pos + Avg_Speed", data=master_driver).fit()
Result_Time_ols.summary()

In [4]:
#alternate file retrieval for more information
folder_path = 'f1 data raw/'

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
csv_list = []

for csv_file in csv_files:
    file_path = os.path.join(folder_path, csv_file)

    track = csv_file[:-9]
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['Result'])
    df['Track'] = track
    csv_list.append(df)

    # vectorizes Driver and Track data
    master_driver = pd.concat(csv_list, ignore_index = True)

In [5]:
#vectorizes Driver and Track information
ohe = OneHotEncoder()
ohe_array = ohe.fit_transform(master_driver[['Driver', 'Track']]).toarray()
column_names = ohe.get_feature_names_out(['Driver', 'Track'])
ohe_df = pd.DataFrame(ohe_array, columns= column_names)

In [6]:
#fitting for models
x = pd.concat([master_driver.drop(columns=['Driver','Track', 'Team', 'Result', 'Finish Pos']), ohe_df], axis = 1)
y = master_driver['Finish Pos']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state = 2)

scaler = StandardScaler()
scaler.fit(x_train)

x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
#linear regression model
lin_reg = LinearRegression()
print(lin_reg.get_params())

lin_reg.fit(x_train, y_train)
print(lin_reg.intercept_)
print(lin_reg.coef_)

print(lin_reg.score(x_test,y_test))

In [7]:
# MLP NN
array_list = []
for m in range(10):
    i_scores = []
    for i in range(10):
        f_scores = []
        for f in range(5):
            mlp_reg = MLPRegressor(hidden_layer_sizes=(2*m + 30, 2*i + 20),verbose=0, activation='relu', solver='adam', max_iter=5000)
            mlp_reg.fit(x_train, y_train)
            f_scores.append(mlp_reg.score(x_test,y_test))
        i_scores.append(mean(f_scores))
    array_list.append(i_scores)
nn_scores = pd.DataFrame(array_list)

In [ ]:
nn_scores.index = ['30', '32', '34', '36', '38', '40', '42', '44', '46', '48']
nn_scores.columns = ['20', '22', '24', '26', '28', '30', '32', '34', '36', '38']
sns.heatmap(nn_scores, cmap = 'coolwarm', annot=True)

In [ ]:
# SVM
# Find the best C for an svm model (this will take a while to run!)
folder_path = 'C:/Users/ammon/Downloads/Exploring_Formula_1_in_the_Ground_Effect_Era/Exploring Formula 1 in the Ground Effect Era/f1 data raw'

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
csv_list = []

for csv_file in csv_files:
    file_path = os.path.join(folder_path, csv_file)

    track = csv_file[:-9]
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['Result'])
    df['Track'] = track
    csv_list.append(df)

    # vectorizes Driver and Track data
    master_driver = pd.concat(csv_list, ignore_index = True)

# Initialize one hot encoder based on driver and track
ohe = OneHotEncoder()
ohe_array = ohe.fit_transform(master_driver[['Driver', 'Track']]).toarray()
column_names = ohe.get_feature_names_out(['Driver', 'Track'])
ohe_df = pd.DataFrame(ohe_array, columns= column_names)

# Predictors and (true) labels
X = pd.concat([master_driver.drop(columns=['Driver','Track', 'Team', 'Result', 'Finish Pos']), ohe_df], axis = 1) #predictors

# Create binary classifier
master_driver['Finish Pos'] = master_driver['Finish Pos'] <= 3
y = master_driver['Finish Pos']    #labels

# iterate through c-values to find the optimal one based on cross-validation score
Cs = range(1,100)
Accuracies = np.zeros(len(Cs))
for i,C in enumerate(Cs):
    svm_classifier = svm.SVC(kernel='rbf', C = C,gamma='auto')
    scores = cross_val_score(estimator = svm_classifier, X = scale(X), y = y, cv = 3, scoring='accuracy')
    Accuracies[i]  = scores.mean()

# plot the accuacies as a function of c-value:
plt.plot(Cs[0:100],Accuracies[0:100])
plt.show()

# print the maximum accuracy value
print("The max accuracy is " + str(max(Accuracies)))